# 🚀 Image Captioning — GPU Model Trainer (Google Colab)

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook will:
1. Verify GPU availability
2. Clone your project from GitHub
3. Install dependencies
4. Mount Google Drive to persist trained model
5. Run the full training pipeline with EarlyStopping & checkpoints
6. Plot training history

## ✅ Step 1 — Verify GPU

In [ ]:
!nvidia smi

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'Num GPUs Available: {len(gpus)}')
for gpu in gpus:
    print(f'  → {gpu}')

if not gpus:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('\n✅ GPU ready!')

Num GPUs Available: 1
  → PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✅ GPU ready!


## 📁 Step 2 — Mount Google Drive

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/imcaption_trained'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'📂 Model will be saved to: {DRIVE_SAVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Model will be saved to: /content/drive/MyDrive/imcaption_trained


## 📦 Step 3 — Clone Repo & Install Dependencies

> 💡 Replace the `REPO_URL` with your actual GitHub repository URL.

In [11]:
import os

REPO_URL = 'https://github.com/divyanshrajput118/imcaption.git' 

if not os.path.exists('/content/imcaption'):
    os.system(f'git clone {REPO_URL} /content/imcaption')
else:
    print('Repo already cloned. Pulling latest...')
    os.system('cd /content/imcaption && git pull')

os.chdir('/content/imcaption')
print(f'Working directory: {os.getcwd()}')

Repo already cloned. Pulling latest...
Working directory: /content/imcaption


In [1]:
!pip install -e . -q
!pip install -r requirements_colab.txt -q
print('✅ Dependencies installed!')

ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_colab.txt'
✅ Dependencies installed!


## 📥 Step 4 — Check / Load Artifacts

> ⚡ If you have artifacts saved in Drive from a previous run, set `COPY_FROM_DRIVE = True`.

In [12]:
!python main.py

Traceback (most recent call last):
  File "/content/imcaption/main.py", line 2, in <module>
    from imgCaption.pipeline.stage_01_data_ingestion import DataIngestionTrainingPipeline
  File "/content/imcaption/src/imgCaption/pipeline/stage_01_data_ingestion.py", line 1, in <module>
    from imgCaption.config.configuration import ConfigurationManager
  File "/content/imcaption/src/imgCaption/config/configuration.py", line 2, in <module>
    from imgCaption.utils.common import read_yaml,create_directories
  File "/content/imcaption/src/imgCaption/utils/common.py", line 7, in <module>
    from ensure import ensure_annotations
  File "/usr/local/lib/python3.12/dist-packages/ensure/__init__.py", line 4, in <module>
    from .main import EnsureError, Ensure, Check, ensure, check, ensure_raises, ensure_raises_regex, ensure_annotations
  File "/usr/local/lib/python3.12/dist-packages/ensure/main.py", line 922, in <module>
    ensure_raises_regex = unittest_case.assertRaisesRegexp
               

In [ ]:
# Run this immediately after !python main.py completes
import shutil, os

# Save the trained model to Drive
shutil.copy(
    '/content/imcaption/artifacts/training/model.h5',
    '/content/drive/MyDrive/imcaption_trained/model.h5'
)
print("✅ Trained model saved to Drive!")


In [8]:
import shutil, os

DRIVE_ARTIFACTS = '/content/drive/MyDrive/imcaption_artifacts'
os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)

# Only copy processed artifacts (skip the large Images folder)
for folder in ['data_transformation', 'prepare_base_model', 'training']:
    src = f'/content/imcaption/artifacts/{folder}'
    dst = f'{DRIVE_ARTIFACTS}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Saved: {folder}")
    else:
        print(f"⚠️ Skipped (not found): {folder}")

⚠️ Skipped (not found): data_transformation
⚠️ Skipped (not found): prepare_base_model
⚠️ Skipped (not found): training


In [7]:
import os

required = [
    'artifacts/data_ingestion/Images',
    'artifacts/data_transformation/vectorizer_data.pkl',
    'artifacts/data_transformation/train_images_captions.pkl',
    'artifacts/prepare_base_model/feature_extractor.h5',
    'artifacts/prepare_base_model/main_model.h5',
]

missing = [p for p in required if not os.path.exists(p)]

if missing:
    print('⚠️  Missing artifacts:')
    for m in missing:
        print(f'  ✗ {m}')
    print('\nEither set COPY_FROM_DRIVE=True below or run the full pipeline.')
else:
    print('✅ All required artifacts found!')

⚠️  Missing artifacts:
  ✗ artifacts/data_ingestion/Images
  ✗ artifacts/data_transformation/vectorizer_data.pkl
  ✗ artifacts/data_transformation/train_images_captions.pkl
  ✗ artifacts/prepare_base_model/feature_extractor.h5
  ✗ artifacts/prepare_base_model/main_model.h5

Either set COPY_FROM_DRIVE=True below or run the full pipeline.


In [ ]:
import shutil

COPY_FROM_DRIVE = True 
DRIVE_ARTIFACTS_DIR = '/content/drive/MyDrive/imcaption_artifacts'

if COPY_FROM_DRIVE:
    if os.path.exists(DRIVE_ARTIFACTS_DIR):
        shutil.copytree(DRIVE_ARTIFACTS_DIR, '/content/imcaption/artifacts', dirs_exist_ok=True)
        print(f'✅ Artifacts copied from {DRIVE_ARTIFACTS_DIR}')
    else:
        print(f'❌ Drive artifacts not found at: {DRIVE_ARTIFACTS_DIR}')
else:
    print('Skipping Drive copy.')

In [ ]:
RUN_FULL_PIPELINE = False  # ← Set True only if starting completely fresh

if RUN_FULL_PIPELINE:
    !python main.py
else:
    print('Skipping full pipeline run.')

## 🏋️ Step 5 — Train the Model on GPU

In [ ]:
import os, pickle
import numpy as np
import tensorflow as tf
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

os.chdir('/content/imcaption')

from imgCaption.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from imgCaption.utils.common import read_yaml, create_directories, load_pkl_file
from imgCaption import logger

print('✅ All imports successful!')
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    image_feex_path: Path
    main_model_path: Path
    vectorizer_path: Path
    images_dir: Path
    trained_model_path: Path
    train_images_captions_path: Path
    SPLIT_SIZE: int
    RANDOM_STATE: int
    MAX_LENGTH: int
    BATCH_SIZE: int

In [ ]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        training = self.config.model_trainer
        base = self.config.prepare_base_model
        dt = self.config.data_transformation
        p = self.params
        create_directories([training.root_dir])
        return ModelTrainerConfig(
            root_dir=Path(training.root_dir),
            image_feex_path=Path(base.image_feex_path),
            main_model_path=Path(base.main_model_path),
            vectorizer_path=Path(dt.vectorizer_path),
            images_dir=Path(training.images_dir),
            trained_model_path=Path(training.trained_model_path),
            train_images_captions_path=Path(dt.train_images_captions_path),
            SPLIT_SIZE=p.SPLIT_SIZE, RANDOM_STATE=p.RANDOM_STATE,
            MAX_LENGTH=p.MAX_LENGTH, BATCH_SIZE=p.BATCH_SIZE,
        )

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.load_vectorizer()
        self.load_feature_extractor()
        self.load_main_model()

    def load_vectorizer(self):
        from_disk = load_pkl_file(self.config.vectorizer_path)
        self.vectorizer = TextVectorization.from_config(from_disk['config'])
        self.vectorizer.adapt(tf.data.Dataset.from_tensor_slices(['dummy']))
        self.vectorizer.set_weights(from_disk['weights'])

    def load_feature_extractor(self):
        self.feature_extractor = load_model(self.config.image_feex_path)

    def load_main_model(self):
        self.main_model = load_model(self.config.main_model_path)

    @staticmethod
    def save_model(path, model):
        model.save(path)
        print(f'💾 Model saved to: {path}')

    def split_dict_by_keys(self, image_ids):
        train, val = train_test_split(image_ids, test_size=self.config.SPLIT_SIZE, random_state=self.config.RANDOM_STATE)
        logger.info(f'Train images: {len(train)}, Val images: {len(val)}')
        return train, val

    def features_ext_dict(self, image_caption_map):
        d = {}
        images = list(image_caption_map.keys())
        batch_size = 64  # Larger on GPU
        for start in tqdm(range(0, len(images), batch_size), desc='Extracting features', unit='batch'):
            batch_names = images[start:start + batch_size]
            batch_imgs = []
            for img_name in batch_names:
                img_path = os.path.join(self.config.images_dir, img_name)
                img = load_img(img_path, target_size=(224, 224))
                img = preprocess_input(img_to_array(img))
                batch_imgs.append(img)
            batch_imgs = np.array(batch_imgs)
            features = self.feature_extractor.predict(batch_imgs, verbose=0)
            for i, image in enumerate(batch_names):
                d[image] = features[i].flatten()
            del batch_imgs, features
        logger.info(f'Feature extraction complete: {len(d)} images')
        return d

    def data_generator(self, image_caption_map, features_map):
        while True:
            X1, X2, Y = [], [], []
            cnt = 0
            for image, captions in image_caption_map.items():
                for cap in captions:
                    cap = self.vectorizer([cap]).numpy()[0]
                    for j in range(1, len(cap)):
                        cur_seq = pad_sequences([cap[:j]], maxlen=self.config.MAX_LENGTH, padding='post')[0]
                        X1.append(features_map[image])
                        X2.append(cur_seq)
                        Y.append(cap[j])
                cnt += 1
                if cnt == self.config.BATCH_SIZE:
                    yield [np.array(X1), np.array(X2)], np.array(Y)
                    X1, X2, Y = [], [], []
                    cnt = 0
            if X1:
                yield [np.array(X1), np.array(X2)], np.array(Y)

    def train(self, epochs=30, drive_save_dir=None):
        train_images_caption = load_pkl_file(self.config.train_images_captions_path)
        image_ids = list(train_images_caption.keys())
        train_keys, val_keys = self.split_dict_by_keys(image_ids)

        train_img_cap_map = {k: train_images_caption[k] for k in train_keys}
        val_img_cap_map = {k: train_images_caption[k] for k in val_keys}

        print('\n🔍 Extracting image features on GPU...')
        features_map = self.features_ext_dict(train_images_caption)
        train_feat_map = {k: features_map[k] for k in train_keys}
        val_feat_map = {k: features_map[k] for k in val_keys}

        train_gen = self.data_generator(train_img_cap_map, train_feat_map)
        val_gen = self.data_generator(val_img_cap_map, val_feat_map)

        self.main_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        best_ckpt_path = str(self.config.trained_model_path)
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
            ModelCheckpoint(best_ckpt_path, monitor='val_loss', save_best_only=True, verbose=1),
        ]

        if drive_save_dir:
            drive_ckpt = os.path.join(drive_save_dir, 'best_model.h5')
            callbacks.append(ModelCheckpoint(drive_ckpt, monitor='val_loss', save_best_only=True, verbose=1))
            print(f'💾 Best model → Drive: {drive_ckpt}')

        steps_per_epoch = len(train_img_cap_map) // self.config.BATCH_SIZE
        validation_steps = len(val_img_cap_map) // self.config.BATCH_SIZE

        print(f'\n🏋️ Training — up to {epochs} epochs')
        print(f'   steps_per_epoch  : {steps_per_epoch}')
        print(f'   validation_steps : {validation_steps}')

        history = self.main_model.fit(
            train_gen, validation_data=val_gen,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            validation_steps=validation_steps,
            callbacks=callbacks
        )

        self.save_model(self.config.trained_model_path, self.main_model)
        if drive_save_dir:
            final_path = os.path.join(drive_save_dir, 'model_final.h5')
            self.save_model(final_path, self.main_model)
            print(f'\n✅ Final model saved to Drive: {final_path}')

        return history

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    history = model_trainer.train(epochs=30, drive_save_dir=DRIVE_SAVE_DIR)
except Exception as e:
    raise e

## 📊 Step 6 — Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss', color='#4F86C6', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#F4A261', lw=2)
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['accuracy'], label='Train Acc', color='#4F86C6', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='#F4A261', lw=2)
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(DRIVE_SAVE_DIR, 'training_history.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Plot saved: {plot_path}')

## 💾 Step 7 — Backup All Artifacts to Drive (Optional)

In [ ]:
import shutil

BACKUP_ARTIFACTS = False  # ← Set True to copy everything to Drive

if BACKUP_ARTIFACTS:
    backup_path = '/content/drive/MyDrive/imcaption_artifacts'
    if os.path.exists(backup_path):
        shutil.rmtree(backup_path)
    shutil.copytree('/content/imcaption/artifacts', backup_path)
    print(f'✅ All artifacts backed up to: {backup_path}')
else:
    print('Skipping artifact backup. Set BACKUP_ARTIFACTS=True to enable.')